## Supplementary Regression Analysis

This notebook includes supplementary regression tests for pricing sensitivity review, including:
- random-rate benchmarking
- generalized linear modeling (GLM)
- comparison with the primary OLS framework

In [ ]:
# Random Rate Regression
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Load Data
file_path = "data/sample_pricing_data.xlsx"
df = pd.read_excel(file_path, sheet_name="PRIMARY DATA")

# Prepare Data
df["DATE BOUND"] = pd.to_datetime(df["DATE BOUND"])
df["MONTH_BOUND"] = df["DATE BOUND"].dt.to_period("M").astype(str)
df["HAZARD_SCORE"] = df["FIRE SCORE"].astype(str)
df["LATITUDE"] = pd.to_numeric(df["LATITUDE"], errors="coerce")
df["LONGITUDE"] = pd.to_numeric(df["LONGITUDE"], errors="coerce")
df["PROPERTY LIMIT"] = pd.to_numeric(df["PROPERTY LIMIT"], errors="coerce")
df["POLICY LIMIT - COV A"] = pd.to_numeric(df["POLICY LIMIT - COV A"], errors="coerce")
df["PREMIUM RATE (%)"] = pd.to_numeric(df["PREMIUM RATE (%)"], errors="coerce")

limit_bins = [0, 1e6, 5e6, 10e6, np.inf]
limit_labels = ["0-1M", "1-5M", "5-10M", ">10M"]

df["TIV_BAND"] = pd.cut(df["PROPERTY LIMIT"], bins=limit_bins, labels=limit_labels)
df["COV_A_BAND"] = pd.cut(df["POLICY LIMIT - COV A"], bins=limit_bins, labels=limit_labels)

df = df.dropna(subset=["PREMIUM RATE (%)", "LATITUDE", "LONGITUDE"]).copy()

print(df.head())
print(df.shape)

In [ ]:
# Create a random-rate benchmark
np.random.seed(42)
df["RANDOM_RATE"] = np.random.uniform(
    df["PREMIUM RATE (%)"].min(),
    df["PREMIUM RATE (%)"].max(),
    len(df)
)

y_random = df["RANDOM_RATE"]
X_base = df[["LATITUDE", "LONGITUDE"]]
X_cat = pd.get_dummies(df[["HAZARD_SCORE", "TIV_BAND", "COV_A_BAND", "MONTH_BOUND"]], drop_first=False)
X_random = pd.concat([X_base, X_cat], axis=1)
X_random = sm.add_constant(X_random).astype(float)

random_model = sm.OLS(y_random, X_random).fit()

print("Random Rate Model")
print("R-squared:", round(random_model.rsquared, 4))
print(random_model.summary().tables[0])

In [ ]:
# Compare random vs actual distribution
plt.figure(figsize=(8, 6))
plt.hist(df["PREMIUM RATE (%)"], bins=20, alpha=0.6, label="Actual Premium Rate")
plt.hist(df["RANDOM_RATE"], bins=20, alpha=0.6, label="Random Rate")
plt.xlabel("Rate")
plt.ylabel("Frequency")
plt.title("Actual vs Random Rate Distribution")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compare random vs actual distribution
plt.figure(figsize=(8, 6))
plt.hist(df["PREMIUM RATE (%)"], bins=20, alpha=0.6, label="Actual Premium Rate")
plt.hist(df["RANDOM_RATE"], bins=20, alpha=0.6, label="Random Rate")
plt.xlabel("Rate")
plt.ylabel("Frequency")
plt.title("Actual vs Random Rate Distribution")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GLM prediction review
glm_pred = glm_model.predict(glm_df)

plt.figure(figsize=(8, 6))
plt.scatter(glm_df["PREMIUM RATE (%)"], glm_pred, alpha=0.6)
plt.plot(
    [glm_df["PREMIUM RATE (%)"].min(), glm_df["PREMIUM RATE (%)"].max()],
    [glm_df["PREMIUM RATE (%)"].min(), glm_df["PREMIUM RATE (%)"].max()],
    "r--"
)
plt.xlabel("Actual PREMIUM RATE")
plt.ylabel("Predicted PREMIUM RATE")
plt.title("GLM Actual vs Predicted")
plt.grid(True)
plt.tight_layout()
plt.show()

## Key Findings

- A random-rate benchmark produces weak explanatory power, which helps validate that the primary model is capturing real structure in the data.
- The GLM specification provides an alternative lens for testing pricing sensitivity.
- Supplementary modeling helps assess robustness across different assumptions and model forms.